# Aula 5,5: teoria, corpus de 8 documentos (Módulos 1 a 9)

In [2]:
ranking <- c("d3","d1","d2","d4","d8","d6","d5","d7")   # saída do BM25, Aula 04
grau <- c(d1 = 1, d2 = 2, d3 = 2, d4 = 0,                # gabarito da Aula 05
          d5 = 0, d6 = 1, d7 = 0, d8 = 0)
relevantes <- names(grau)[grau == 2]                     # relevante = grau 2
rel <- as.integer(ranking %in% relevantes)               # 1 se ranking[i] é relevante
rel

[1] 1 0 1 0 0 0 0 0

In [3]:
R <- length(relevantes)   # 2 relevantes existem no corpus
k <- seq_along(ranking)   # posições de corte: 1, 2, ..., 8
acertos <- cumsum(rel)    # soma acumulada: quantos relevantes até a posição k
precisao <- acertos / k   # acertos sobre o que foi MOSTRADO
recall   <- acertos / R   # acertos sobre o TOTAL de relevantes

In [4]:
tabela <- round(rbind(P_at_k = precisao, R_at_k = recall), 3)   # duas linhas, 8 colunas
colnames(tabela) <- paste0("k=", k)                              # rótulos k=1 ... k=8
tabela

,k=1,k=2,k=3,k=4,k=5,k=6,k=7,k=8
P_at_k,1.0,0.5,0.667,0.5,0.4,0.333,0.286,0.25
R_at_k,0.5,0.5,1.000,1.0,1.0,1.000,1.000,1.00


In [5]:
precisao[rel == 1]                  # as precisões nas posições com relevante
ap <- sum(precisao[rel == 1]) / R   # soma dividida por R, o total de relevantes
round(ap, 3)

[1] 1.0000000 0.6666667

[1] 0.833

In [6]:
which(rel == 1)                 # posições de TODOS os relevantes
mrr <- 1 / which(rel == 1)[1]   # [1] pega só a primeira
mrr

[1] 1 3

[1] 1

In [7]:
dcg <- function(r) sum(r / log2(seq_along(r) + 1))    # desconta cada ganho pela posição e soma
ndcg <- dcg(rel) / dcg(sort(rel, decreasing = TRUE))  # ideal = os MESMOS ganhos, ordenados
round(ndcg, 3)

[1] 0.92

In [8]:
g <- grau[ranking]   # o grau de cada documento, NA ORDEM em que o sistema os devolveu
g

d3 d1 d2 d4 d8 d6 d5 d7 
 2  1  2  0  0  1  0  0

In [9]:
g <- grau[ranking]                                  # graus na ordem do ranking
ndcg_g <- dcg(g) / dcg(sort(g, decreasing = TRUE))  # mesma função, agora com graus
round(ndcg_g, 3)

[1] 0.951

In [10]:
resumo <- c(P_at_3 = precisao[[3]], AP = ap, MRR = mrr,   # [[3]]: o valor, sem o nome grudado
            nDCG_bin = ndcg, nDCG_grad = ndcg_g)
round(resumo, 3)

P_at_3        AP       MRR  nDCG_bin nDCG_grad 
    0.667     0.833     1.000     0.920     0.951

# Aula 5,5, Parte D: métricas no gabarito da equipe (Módulos 10 a 12)
Daqui em diante as cinco métricas (P@3, AP, RR, nDCG binário e nDCG graduado) são aplicadas ao gabarito do nosso corpus, comparando o cosseno (TF-IDF) e o BM25. As sete consultas de **desenvolvimento** passam pelas contas; o conjunto de **teste** fica lacrado.

In [11]:
system("git clone https://github.com/vicleme/PI3-FFV.git")     # baixa o projeto para a sessão
setwd("PI3-FFV")                                               # entra na raiz do projeto
file.exists("estrutura/banco-de-dados/qrels_consolidado.csv")  # TRUE se achou o gabarito

[1] TRUE

In [12]:
q <- read.csv("estrutura/banco-de-dados/qrels_consolidado.csv")   # o gabarito do grupo
head(q)                                                            # as primeiras linhas
nrow(q)                                                            # número de linhas

,consulta,documento,grau
,<chr>,<chr>,<int>
1,q01,Cubatao_p1,0
2,q02,Cubatao_p1,0
3,q03,Cubatao_p1,0
4,q04,Cubatao_p1,0
5,q05,Cubatao_p1,0
6,q06,Cubatao_p1,0


[1] 630

In [13]:
linhas <- q[q$consulta == "q02", ]                  # só as linhas dessa consulta
grau   <- setNames(linhas$grau, linhas$documento)   # vetor nomeado: documento -> grau
length(grau)                                         # quantos elementos
head(grau)                                           # os seis primeiros

[1] 63

Cubatao_p1 Cubatao_p10 Cubatao_p11 Cubatao_p12 Cubatao_p13 Cubatao_p14 
          0           0           0           0           0           0

In [14]:
sum(grau == 2)   # quantos parágrafos têm grau 2 na q02
sum(grau == 1)   # quantos têm grau 1 na q02

[1] 1

[1] 1

In [15]:
rankings <- readRDS("estrutura/banco-de-dados/rankings.rds")   # os rankings dos 3 modelos
ranking_bm25 <- rankings[["BM25"]][["q02"]]                     # ranking do BM25 na q02
ranking_cos  <- rankings[["TF-IDF"]][["q02"]]                   # ranking do cosseno na q02
relevantes <- names(grau)[grau >= 2]                            # o limiar da equipe
relevantes

[1] "Santos_p14"

## Módulo 10 (continuação): o `rel` da q02, para os dois sistemas
Limiar da equipe: **grau ≥ 2**, para os dois sistemas e todas as consultas. Ele foi decidido antes de calcular qualquer métrica. Parágrafo que não está no gabarito (não julgado) conta como grau 0, pela regra do *pooling*.

In [16]:
rel_bm25 <- as.integer(ranking_bm25 %in% relevantes)   # 1 se o parágrafo do ranking é relevante
rel_cos  <- as.integer(ranking_cos  %in% relevantes)   # o mesmo, para o cosseno
length(rel_bm25)         # quantas posições tem o ranking
which(rel_bm25 == 1)     # em que posição o relevante caiu no BM25
which(rel_cos == 1)      # e no cosseno

[1] 63

[1] 2

[1] 3

Dois números para conferir: $R$, o total de relevantes da q02 com este limiar, e quantos desses o BM25 pôs nas três primeiras posições.

In [17]:
length(relevantes)        # R
sum(rel_bm25[1:3])        # relevantes nas 3 primeiras posições do BM25

[1] 1

[1] 1

## Módulo 11: as cinco métricas na q02, BM25 e cosseno
Primeiro as três que só usam o `rel`: P@3, AP e RR.

In [18]:
R <- length(relevantes)                                         # total de relevantes da q02

precisao_bm25 <- cumsum(rel_bm25) / seq_along(rel_bm25)         # P@k do BM25, k = 1..63
precisao_cos  <- cumsum(rel_cos)  / seq_along(rel_cos)          # P@k do cosseno

ap_bm25 <- sum(precisao_bm25[rel_bm25 == 1]) / R                # AP: precisões nas posições com relevante, divididas por R
ap_cos  <- sum(precisao_cos[rel_cos == 1])   / R

rr_bm25 <- 1 / which(rel_bm25 == 1)[1]                          # RR: 1 / posição do primeiro relevante
rr_cos  <- 1 / which(rel_cos == 1)[1]

round(c(P_at_3 = precisao_bm25[[3]], AP = ap_bm25, RR = rr_bm25), 3)   # BM25
round(c(P_at_3 = precisao_cos[[3]],  AP = ap_cos,  RR = rr_cos), 3)    # cosseno

P_at_3     AP     RR 
 0.333  0.500  0.500

P_at_3     AP     RR 
 0.333  0.333  0.333

nDCG binário. O nDCG soma o ganho de cada posição multiplicado por um **fator de desconto** $1/\log_2(i+1)$: ele vale 1 na posição 1 e diminui nas seguintes, então um relevante lá embaixo conta pouco. A soma é dividida pela mesma conta no ranking ideal (IDCG). Com um único relevante, o IDCG é 1 (relevante na posição 1, fator de desconto 1).
- BM25: relevante na posição 2, DCG $= 1/\log_2 3 \approx 0{,}631$
- Cosseno: relevante na posição 3, DCG $= 1/\log_2 4 = 0{,}5$

In [19]:
dcg <- function(r) sum(r / log2(seq_along(r) + 1))   # ganho × fator de desconto, somado (a mesma do Módulo 7)

ndcg_bm25 <- dcg(rel_bm25) / dcg(sort(rel_bm25, decreasing = TRUE))   # ideal = os MESMOS ganhos, ordenados
ndcg_cos  <- dcg(rel_cos)  / dcg(sort(rel_cos,  decreasing = TRUE))

round(c(BM25 = ndcg_bm25, cosseno = ndcg_cos), 3)

BM25 cosseno 
  0.631   0.500

nDCG graduado. Aqui entram os parágrafos de grau 1.
`is.na(x)` testa se cada valor é `NA` (ausente). `grau[ranking]` devolve `NA` para parágrafo fora do gabarito, e pela regra do *pooling* esse parágrafo vale grau 0.

In [20]:
g_bm25 <- grau[ranking_bm25]                  # graus na ordem do ranking do BM25
g_bm25[is.na(g_bm25)] <- 0                    # não julgado = grau 0 (pooling)
g_cos  <- grau[ranking_cos]                   # o mesmo para o cosseno
g_cos[is.na(g_cos)] <- 0

ndcg_g_bm25 <- dcg(g_bm25) / dcg(sort(g_bm25, decreasing = TRUE))
ndcg_g_cos  <- dcg(g_cos)  / dcg(sort(g_cos,  decreasing = TRUE))
ndcg_g_bm25
ndcg_g_cos

[1] 0.562525

[1] 0.4629938

Por que o graduado ficou **menor** que o binário? Depende de onde está o grau 1. No ranking ideal ele entra na posição 2 (fator de desconto 0,631), mas nos rankings reais está na posição 23 (fator de desconto cerca de 0,22), onde quase não soma. O ideal ganha mais com ele do que o ranking real, e a razão cai.

In [21]:
table(grau)                                      # quantos parágrafos há de cada grau na q02
grau[ranking_bm25][1:5]                          # graus das 5 primeiras posições do BM25
grau[ranking_cos][1:5]                           # e do cosseno
which(ranking_bm25 == names(grau)[grau == 1])    # posição do grau 1 no BM25
which(ranking_cos  == names(grau)[grau == 1])    # e no cosseno

grau
 0  1  2 
61  1  1 

Santos_p11  Santos_p14  Guaruja_p2  Santos_p12 Guaruja_p17 
          0           2           0           0           0

Guaruja_p2  Santos_p11  Santos_p14  Santos_p12 Guaruja_p15 
          0           0           2           0           0

[1] 23

[1] 23

### Tabela da q02

In [22]:
tabela_q02 <- cbind(
  cosseno = c(P_at_3 = precisao_cos[[3]],  AP = ap_cos,  RR = rr_cos,  nDCG_bin = ndcg_cos,  nDCG_grad = ndcg_g_cos),
  BM25    = c(P_at_3 = precisao_bm25[[3]], AP = ap_bm25, RR = rr_bm25, nDCG_bin = ndcg_bm25, nDCG_grad = ndcg_g_bm25)
)
round(tabela_q02, 3)

,cosseno,BM25
P_at_3,0.333,0.333
AP,0.333,0.500
RR,0.333,0.500
nDCG_bin,0.500,0.631
nDCG_grad,0.463,0.563


### Diferença entre os sistemas na q02
BM25 menos cosseno, por métrica. A maior diferença está no AP e no RR (0,167), que dependem da posição do relevante: 2º no BM25 e 3º no cosseno. O P@3 não vê isso, porque nos dois sistemas o relevante está entre os três primeiros.

In [23]:
round(tabela_q02[, "BM25"] - tabela_q02[, "cosseno"], 3)   # AP e RR diferem mais; P@3 empata

P_at_3        AP        RR  nDCG_bin nDCG_grad 
    0.000     0.167     0.167     0.131     0.100

## q03 (R = 3)
A q03 tem três relevantes. A célula carrega o gabarito e os rankings dela, no mesmo formato da q02, e mostra em que posições os relevantes caem em cada sistema.

In [24]:
grau <- setNames(q[q$consulta == "q03", ]$grau, q[q$consulta == "q03", ]$documento)  # gabarito da q03
relevantes <- names(grau)[grau >= 2]                    # limiar da equipe
ranking_bm25 <- rankings[["BM25"]][["q03"]]
ranking_cos  <- rankings[["TF-IDF"]][["q03"]]
which(ranking_bm25 %in% relevantes)                     # posições dos relevantes no BM25
which(ranking_cos  %in% relevantes)                     # e no cosseno

[1] 1 2 3

[1] 1 2 3

In [25]:
which(ranking_bm25 %in% names(grau)[grau == 1])   # posições dos grau 1 no BM25
which(ranking_cos  %in% names(grau)[grau == 1])   # e no cosseno

[1] 4 7

[1] 4 7

## As cinco métricas numa função só
A q02 foi montada linha a linha. Para não repetir o mesmo bloco em cada consulta, a cadeia inteira vira uma função, do mesmo tipo da `dcg`: recebe um ranking e o gabarito e devolve as cinco métricas. Ela usa variáveis locais, então **não sobrescreve** o `grau` e o `relevantes` do notebook.

In [26]:
metricas_5 <- function(ranking, grau) {
  relevantes <- names(grau)[grau >= 2]               # o limiar da equipe
  R <- length(relevantes)
  rel <- as.integer(ranking %in% relevantes)         # 1 se o parágrafo do ranking é relevante
  precisao <- cumsum(rel) / seq_along(rel)           # P@k
  g <- grau[ranking]                                 # graus na ordem do ranking
  g[is.na(g)] <- 0                                   # não julgado = grau 0 (pooling)
  c(P_at_3    = precisao[[3]],
    AP        = sum(precisao[rel == 1]) / R,
    RR        = 1 / which(rel == 1)[1],
    nDCG_bin  = dcg(rel) / dcg(sort(rel, decreasing = TRUE)),
    nDCG_grad = dcg(g)   / dcg(sort(g,   decreasing = TRUE)))
}

tabela_consulta <- function(consulta) {
  linhas <- q[q$consulta == consulta, ]                       # gabarito da consulta
  grau <- setNames(linhas$grau, linhas$documento)
  cbind(cosseno = metricas_5(rankings[["TF-IDF"]][[consulta]], grau),
        BM25    = metricas_5(rankings[["BM25"]][[consulta]],   grau))
}

posicoes <- function(consulta) {                              # onde caíram os relevantes
  linhas <- q[q$consulta == consulta, ]
  relevantes <- linhas$documento[linhas$grau >= 2]
  list(cosseno = which(rankings[["TF-IDF"]][[consulta]] %in% relevantes),
       BM25    = which(rankings[["BM25"]][[consulta]]   %in% relevantes))
}

**Conferência da função:** na q02 ela precisa dar exatamente a tabela montada à mão. O `all.equal` compara as duas tabelas e devolve `TRUE` quando são iguais.

In [27]:
all.equal(tabela_consulta("q02"), tabela_q02)   # TRUE = a função reproduz a conta manual
round(tabela_consulta("q02"), 3)

[1] TRUE

,cosseno,BM25
P_at_3,0.333,0.333
AP,0.333,0.500
RR,0.333,0.500
nDCG_bin,0.500,0.631
nDCG_grad,0.463,0.563


### q03 (R = 3), pela função

In [28]:
posicoes("q03")                      # posições dos 3 relevantes em cada sistema
round(tabela_consulta("q03"), 3)     # as cinco métricas

$cosseno
[1] 1 2 3

$BM25
[1] 1 2 3

,cosseno,BM25
P_at_3,1.000,1.000
AP,1.000,1.000
RR,1.000,1.000
nDCG_bin,1.000,1.000
nDCG_grad,0.989,0.989


## As outras consultas de desenvolvimento
O conjunto de **teste** (`q01`, `q04`, `q08`) está lacrado: não entra nestas contas e não foi usado para escolher nada. As sete consultas de desenvolvimento (`q02`, `q03`, `q05`, `q06`, `q07`, `q09`, `q10`) passam pela mesma função.

In [29]:
split <- read.csv("estrutura/banco-de-dados/split_teste_desenvolvimento.csv")   # quem é desenvolvimento e quem é teste
dev <- split$consulta[split$conjunto == "desenvolvimento"]                       # só as de desenvolvimento
dev                                                                              # q02 q03 q05 q06 q07 q09 q10
resultados <- setNames(lapply(dev, tabela_consulta), dev)                        # uma tabela por consulta

[1] "q02" "q03" "q05" "q06" "q07" "q09" "q10"

### q05

In [30]:
posicoes("q05")
round(resultados[["q05"]], 3)

$cosseno
[1] 21

$BM25
[1] 21

,cosseno,BM25
P_at_3,0.000,0.000
AP,0.048,0.048
RR,0.048,0.048
nDCG_bin,0.224,0.224
nDCG_grad,0.266,0.266


Na q05 o único relevante (grau 2) está na posição 21 nos dois sistemas, e o nDCG graduado (0,266) saiu **maior** que o binário (0,224). A diferença vem dos parágrafos de grau 1. Esta célula acha as posições deles nos dois rankings.

In [31]:
linhas05 <- q[q$consulta == "q05", ]                              # gabarito da q05
grau1 <- linhas05$documento[linhas05$grau == 1]                   # os parágrafos de grau 1
which(rankings[["BM25"]][["q05"]] %in% grau1)                     # posições dos grau 1 no BM25
which(rankings[["TF-IDF"]][["q05"]] %in% grau1)                   # e no cosseno

[1] 35 37

[1] 35 37

Para ver quanto cada grau contribui, a conta do nDCG graduado é separada em duas partes, com o fator de desconto $1/\log_2(i+1)$ de cada posição:
- **grau 2 sozinho** (posição 21, contra o ideal na posição 1): 0,224, o mesmo valor do nDCG binário;
- **os dois grau 1 sozinhos** (posições 35 e 37, contra o ideal nas posições 2 e 3): 0,340;
- **graduado** (as duas partes somadas, ganho e ideal): 0,266. Fica entre os dois valores, mais perto do grau 2, porque o ideal do grau 2 (2,0) é maior que o dos grau 1 (cerca de 1,13).

In [32]:
real_2  <- 2 / log2(21 + 1)                       # grau 2 na posição 21
ideal_2 <- 2 / log2(1 + 1)                        # grau 2 no ideal: posição 1
real_1  <- 1/log2(35 + 1) + 1/log2(37 + 1)        # os dois grau 1 nas posições 35 e 37
ideal_1 <- 1/log2(2 + 1)  + 1/log2(3 + 1)         # os dois grau 1 no ideal: posições 2 e 3
round(c(prova_grau2 = real_2/ideal_2, prova_grau1 = real_1/ideal_1,
        graduado = (real_2 + real_1)/(ideal_2 + ideal_1)), 3)

prova_grau2 prova_grau1    graduado 
      0.224       0.340       0.266

### q06

In [33]:
posicoes("q06")
round(resultados[["q06"]], 3)

$cosseno
[1] 1

$BM25
[1] 1

,cosseno,BM25
P_at_3,0.333,0.333
AP,1.000,1.000
RR,1.000,1.000
nDCG_bin,1.000,1.000
nDCG_grad,1.000,1.000


### q07

In [34]:
posicoes("q07")
round(resultados[["q07"]], 3)

$cosseno
[1] 11 12

$BM25
[1] 11 12

,cosseno,BM25
P_at_3,0.000,0.000
AP,0.129,0.129
RR,0.091,0.091
nDCG_bin,0.337,0.337
nDCG_grad,0.337,0.337


### q09

In [35]:
posicoes("q09")
round(resultados[["q09"]], 3)

$cosseno
[1] 2 4

$BM25
[1] 2 4

,cosseno,BM25
P_at_3,0.333,0.333
AP,0.500,0.500
RR,0.500,0.500
nDCG_bin,0.651,0.651
nDCG_grad,0.651,0.651


### q10

In [36]:
posicoes("q10")
round(resultados[["q10"]], 3)

$cosseno
[1] 1

$BM25
[1] 1

,cosseno,BM25
P_at_3,0.333,0.333
AP,1.000,1.000
RR,1.000,1.000
nDCG_bin,1.000,1.000
nDCG_grad,1.000,1.000


## Resumo das 7 consultas
Diferença **BM25 menos cosseno**, por métrica (linhas) e consulta (colunas). Positivo = BM25 na frente.

In [37]:
dif <- sapply(resultados, function(t) t[, "BM25"] - t[, "cosseno"])   # 5 métricas x 7 consultas
round(dif, 3)

,q02,q03,q05,q06,q07,q09,q10
P_at_3,0.000,0,0,0,0,0,0
AP,0.167,0,0,0,0,0,0
RR,0.167,0,0,0,0,0,0
nDCG_bin,0.131,0,0,0,0,0,0
nDCG_grad,0.100,0,0,0,0,0,0


Em quantas consultas cada sistema ganhou, por métrica (empate = notas iguais):

In [38]:
placar <- cbind(BM25    = rowSums(dif >  1e-9),          # consultas em que o BM25 ficou na frente
                cosseno = rowSums(dif < -1e-9),          # consultas em que o cosseno ficou na frente
                empate  = rowSums(abs(dif) <= 1e-9))     # consultas empatadas
placar

,BM25,cosseno,empate
P_at_3,0,0,7
AP,1,0,6
RR,1,0,6
nDCG_bin,1,0,6
nDCG_grad,1,0,6


A média sobre as consultas: a linha `AP` é o **MAP**, a linha `RR` é o **MRR**.

In [39]:
medias <- Reduce("+", resultados) / length(resultados)   # soma as 7 tabelas e divide por 7
round(medias, 3)

,cosseno,BM25
P_at_3,0.333,0.333
AP,0.573,0.597
RR,0.567,0.591
nDCG_bin,0.673,0.692
nDCG_grad,0.672,0.687


### Por que a q02 separou os sistemas?
A q02 ("praias santos quantidade") é a única consulta em que BM25 e cosseno se separam. O único relevante é `Santos_p14`. No cosseno, `Guaruja_p2` e `Santos_p11` ficam na frente dele; no BM25, só `Santos_p11`. Esta célula olha o texto desses três parágrafos: tamanho em caracteres e quantas vezes aparecem as palavras "praias" e "santos". A última linha traz o tamanho médio dos 63 parágrafos, para comparação.

In [40]:
par <- read.csv("estrutura/corpus/paragrafos.csv", encoding = "UTF-8")      # um parágrafo por linha
alvo <- c("Santos_p14", "Guaruja_p2", "Santos_p11")                          # o relevante e os dois que disputaram posição com ele
vistos <- par[match(alvo, par$doc_id), ]                                     # as 3 linhas, na ordem de 'alvo'
txt <- tolower(vistos$texto)
conta <- function(padrao, x) lengths(regmatches(x, gregexpr(padrao, x)))     # quantas vezes o padrão aparece em cada texto
data.frame(doc_id = vistos$doc_id,
           n_caracteres = vistos$n_caracteres,
           praias = conta("praias", txt),
           santos = conta("santos", txt))
mean(par$n_caracteres)    # tamanho médio dos 63 parágrafos

doc_id,n_caracteres,praias,santos
<chr>,<int>,<int>,<int>
Santos_p14,250,1,1
Guaruja_p2,553,3,0
Santos_p11,338,1,3


[1] 441.5714

In [41]:
# --- 1. corpus e tokenizador: os mesmos do projeto ---------------------------------
par  <- read.csv("estrutura/corpus/paragrafos.csv", encoding = "UTF-8")   # um parágrafo por linha
docs <- setNames(par$texto, par$doc_id)                                   # texto de cada parágrafo, com o doc_id como nome
linhas <- readLines("estrutura/codigo/03-corpus-cidades-tfidf-busca.R", encoding = "UTF-8")
ini <- grep("^stopwords_pt <- c\\(", linhas)                              # onde começa a lista de stopwords
fim <- grep("^tokens_cidades <-", linhas) - 1                             # a linha antes de o script usar o tokenizador
eval(parse(text = linhas[ini:fim]))                                       # define stopwords_pt e tokenizar_limpo()

# --- 2. contagens que o BM25 usa -----------------------------------------------------
tokens_par <- lapply(docs, tokenizar_limpo)                               # palavras de cada parágrafo, já limpas
vocab <- sort(unique(unlist(tokens_par)))                                 # vocabulário do corpus
tf <- sapply(tokens_par, function(tk) as.integer(table(factor(tk, levels = vocab))))   # quantas vezes cada palavra aparece em cada parágrafo
rownames(tf) <- vocab
dl    <- sapply(tokens_par, length)                                       # tamanho de cada parágrafo, em palavras
avgdl <- mean(dl)                                                         # tamanho médio
N  <- ncol(tf)                                                            # número de parágrafos
df <- rowSums(tf > 0)                                                     # em quantos parágrafos cada palavra aparece
idf <- log((N - df + 0.5) / (df + 0.5) + 1)                               # IDF do BM25

# --- 3. o BM25, com k1 (saturação) e b (normalização de tamanho) como argumentos ------
bm25_pontos <- function(consulta, k1 = 1.2, b = 0.75) {
  termos <- tokenizar_limpo(consulta)
  termos <- termos[termos %in% vocab]                                     # descarta palavra fora do vocabulário
  sapply(colnames(tf), function(d) {
    f <- tf[termos, d]                                                    # frequência de cada termo da consulta neste parágrafo
    K <- k1 * (1 - b + b * dl[d] / avgdl)                                 # penalidade de tamanho
    sum(idf[termos] * f * (k1 + 1) / (f + K))                             # soma sobre os termos
  })
}
bm25 <- function(consulta, k1 = 1.2, b = 0.75) sort(bm25_pontos(consulta, k1, b), decreasing = TRUE)   # ranking, do maior para o menor

# --- 4. confere: o padrão reproduz o ranking do BM25 que já usamos --------------------
consulta <- "praias santos quantidade"                                    # texto da q02
all.equal(names(bm25(consulta))[1:10], as.character(rankings[["BM25"]][["q02"]])[1:10])   # TRUE = mesmas 10 primeiras posições

# --- 5. liga e desliga cada peça e vê onde caem os três parágrafos --------------------
# Sem a normalização de tamanho, parágrafos com as mesmas contagens empatam em pontos.
# Por isso a posição aparece como faixa (melhor–pior) quando há empate.
alvo <- c("Guaruja_p2", "Santos_p11", "Santos_p14")                       # Santos_p14 é o relevante
variantes <- list("padrão (k1 = 1,2; b = 0,75)"       = c(1.2,  0.75),
                  "sem normalização de tamanho (b = 0)" = c(1.2,  0),
                  "sem saturação (k1 = 1000)"          = c(1000, 0.75),
                  "sem nenhuma das duas"               = c(1000, 0))
faixa <- function(p) {
  pontos <- round(bm25_pontos(consulta, p[1], p[2]), 10)                  # arredonda para o empate ser detectado
  melhor <- rank(-pontos, ties.method = "min")[alvo]                      # posição se ganhar todos os desempates
  pior   <- rank(-pontos, ties.method = "max")[alvo]                      # posição se perder todos os desempates
  ifelse(melhor == pior, melhor, paste0(melhor, "–", pior))
}
posicoes_var <- t(sapply(variantes, faixa))
colnames(posicoes_var) <- alvo
noquote(posicoes_var)

[1] TRUE

                                    Guaruja_p2 Santos_p11 Santos_p14
padrão (k1 = 1,2; b = 0,75)         3          1          2         
sem normalização de tamanho (b = 0) 2          1          4–5       
sem saturação (k1 = 1000)           2          1          3         
sem nenhuma das duas                1          2          6–7       

A tabela acima mostra a posição de cada um dos três parágrafos em cada variante do BM25 (faixa, como "4–5", quando há empate em pontos). Lendo por coluna:
- **`Santos_p14` (o relevante):** é 2º no padrão e piora ao desligar qualquer peça: 3º sem saturação, 4º–5º sem normalização de tamanho, 6º–7º sem as duas.
- **`Guaruja_p2` (longo, repete "praias"):** é 3º no padrão e sobe para 2º ao desligar uma peça e para 1º ao desligar as duas.
- A normalização de tamanho protege o relevante mais que a saturação, e as duas juntas dão a ordem do padrão. Sem elas, o BM25 passa a se comportar como o cosseno nesta consulta: o parágrafo longo que repete "praias" ganha do curto que traz "praias" e "santos".

## Conclusão da equipe
- **Limiar e motivo:** grau ≥ 2, definido antes de calcular qualquer métrica e igual para os dois sistemas e todas as consultas. O grau 1 "toca o assunto mas não responde", então não conta como relevante; ele continua pesando no nDCG graduado, que usa os graus.
- **Quem ficou na frente, e em quantas consultas:** nas 7 consultas de desenvolvimento, o BM25 ficou na frente em 1 (a q02), em AP, RR, nDCG binário e nDCG graduado. Nas outras 6 houve empate, e o cosseno não ganhou em nenhuma. No P@3 houve empate nas 7. Médias (cosseno / BM25): MAP 0,573 / 0,597; MRR 0,567 / 0,591; nDCG binário 0,673 / 0,692; nDCG graduado 0,672 / 0,687; P@3 0,333 nos dois.
- **Consulta em que os sistemas diferem, e por quê:** a q02 ("praias santos quantidade"). O único relevante, `Santos_p14`, fica na posição 2 no BM25 e na 3 no cosseno. À frente dele, no cosseno, está `Guaruja_p2`, um parágrafo longo (553 caracteres, contra a média de 441,6) que repete "praias" três vezes e não menciona "santos". O BM25 tem duas defesas: a saturação (repetir a mesma palavra rende cada vez menos) e a normalização de tamanho (parágrafo longo é penalizado). O teste que liga e desliga cada peça confirma: sem elas, `Santos_p14` cai para 3º, 4º–5º e 6º–7º, e `Guaruja_p2` sobe ao 1º; a normalização de tamanho é a que mais protege o relevante. Isso vale para esta consulta, e não generalizamos a partir dela.
- **Ressalva:** as 6 consultas empatadas empatam porque os relevantes caem nas mesmas posições nos dois sistemas. São consultas fáceis, quatro delas com um só relevante, e um corpus pequeno não discrimina. O empate mostra que a medida não separa os sistemas nessas consultas, e não que eles são equivalentes. A vantagem do BM25 vem de uma única consulta e as diferenças nas médias são pequenas; dizer se isso é mais do que acaso exige um teste estatístico (Aula 16). As consultas de teste (`q01`, `q04`, `q08`) seguem lacradas.